## 1. Load

In [1]:
import pandas as pd
import numpy as np

In [3]:
df_raw = pd.read_csv("../data/raw/DataCoSupplyChainDataset.csv", encoding="latin-1")
print(df_raw.shape)

(180519, 53)


## 2. Standardize names


In [4]:
df = df_raw.copy()
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
              .str.replace("(", "", regex=False)
              .str.replace(")", "", regex=False))

In [5]:
changed = [(old, new) for old, new in zip(df_raw.columns, df.columns) if old != new]
print(len(changed), "of", df.shape[1], "names changed")
print("Names unique:", df.columns.is_unique)
for old, new in changed[:3]:
    print(old, "->", new)

53 of 53 names changed
Names unique: True
Type -> type
Days for shipping (real) -> days_for_shipping_real
Days for shipment (scheduled) -> days_for_shipment_scheduled


## 3. Fix text in order_region


In [6]:
before = df["order_region"].nunique()
df["order_region"] = df["order_region"].str.strip().str.replace(r"\s+", " ", regex=True)
print("Regions before:", before, "after:", df["order_region"].nunique())
print(sorted(df["order_region"].unique()))

Regions before: 23 after: 23
['Canada', 'Caribbean', 'Central Africa', 'Central America', 'Central Asia', 'East Africa', 'East of USA', 'Eastern Asia', 'Eastern Europe', 'North Africa', 'Northern Europe', 'Oceania', 'South America', 'South Asia', 'South of USA', 'Southeast Asia', 'Southern Africa', 'Southern Europe', 'US Center', 'West Africa', 'West Asia', 'West of USA', 'Western Europe']


## 4. Parse the order date

In [7]:
df["order_date_dateorders"] = pd.to_datetime(df["order_date_dateorders"], format="%m/%d/%Y %H:%M")
print("Type:", df["order_date_dateorders"].dtype)
print("First order:", df["order_date_dateorders"].min())
print("Last order:", df["order_date_dateorders"].max())
dates_per_order = df.groupby("order_id")["order_date_dateorders"].nunique()
print("Orders with more than one date:", (dates_per_order > 1).sum())

Type: datetime64[us]
First order: 2015-01-01 00:00:00
Last order: 2018-01-31 23:38:00
Orders with more than one date: 0


## 5. Remove orders that never shipped

In [8]:
never_shipped = df["order_status"].isin(["CANCELED", "SUSPECTED_FRAUD"])
print(pd.crosstab(df["delivery_status"], never_shipped, colnames=["never_shipped"]))
print("Orders with more than one status:", (df.groupby("order_id")["order_status"].nunique() > 1).sum())

never_shipped      False  True 
delivery_status                
Advance shipping   41592      0
Late delivery      98977      0
Shipping canceled      0   7754
Shipping on time   32196      0
Orders with more than one status: 0


In [9]:
rows_before = len(df)
df = df[~never_shipped].reset_index(drop=True)
print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Removed:", rows_before - len(df))
print(df["late_delivery_risk"].value_counts())

Rows before: 180519
Rows after: 172765
Removed: 7754
late_delivery_risk
1    98977
0    73788
Name: count, dtype: int64


## 6. Check the target

In [10]:
target = df["late_delivery_risk"]
print("Type:", target.dtype)
print("Values:", sorted(target.unique()))
print("Missing:", target.isnull().sum())
print("Late share:", round(target.mean() * 100, 1), "%")
rule = (df["days_for_shipping_real"] > df["days_for_shipment_scheduled"]).astype(int)
print("Rule match:", (rule == target).mean())

Type: int64
Values: [np.int64(0), np.int64(1)]
Missing: 0
Late share: 57.3 %
Rule match: 1.0


## 7. Check for duplicates

In [11]:
print("Duplicate rows:", df.duplicated().sum())
print("Item ids unique:", df["order_item_id"].is_unique)

Duplicate rows: 0
Item ids unique: True


In [12]:
duplicate_column_pairs = []

for x in range(df.shape[1]):
    for y in range(x + 1, df.shape[1]):

        if df.iloc[:, x].equals(df.iloc[:, y]):
            duplicate_column_pairs.append(
                (df.columns[x], df.columns[y])
            )

print("Duplicate column pairs:")
for col1, col2 in duplicate_column_pairs:
    print(f"{col1}  <-->  {col2}")

Duplicate column pairs:
benefit_per_order  <-->  order_profit_per_order
sales_per_customer  <-->  order_item_total
category_id  <-->  product_category_id
customer_email  <-->  customer_password
customer_id  <-->  order_customer_id
order_item_cardprod_id  <-->  product_card_id
order_item_product_price  <-->  product_price


In [13]:
keep = [
    "order_id", "order_date_dateorders", "late_delivery_risk",
    "type", "days_for_shipment_scheduled", "shipping_mode",
    "market", "order_region", "order_country", "order_state", "order_city",
    "customer_segment", "category_name", "department_name",
    "benefit_per_order", "sales_per_customer", "sales", "order_item_product_price",
    "order_item_quantity", "order_item_discount", "order_item_discount_rate",
    "order_item_profit_ratio",
]
dropped = [c for c in df.columns if c not in keep]
df = df[keep].copy()

In [14]:
print("Dropped:", len(dropped))
print(dropped)
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

Dropped: 31
['days_for_shipping_real', 'delivery_status', 'category_id', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'latitude', 'longitude', 'order_customer_id', 'order_item_cardprod_id', 'order_item_id', 'order_item_total', 'order_profit_per_order', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_dateorders']
Shape: (172765, 22)
Missing values: 0


## Save

In [15]:
df.to_csv("../data/interim/clean_full.csv", index=False)

In [16]:
check = pd.read_csv("../data/interim/clean_full.csv", parse_dates=["order_date_dateorders"])
print("Saved shape:", df.shape)
print("Reloaded shape:", check.shape)
print("Same columns:", list(check.columns) == list(df.columns))
print("Date type:", check["order_date_dateorders"].dtype)
print("First order:", check["order_date_dateorders"].min())
print("Last order:", check["order_date_dateorders"].max())
print("Orders:", check["order_id"].nunique())
print("Late share:", round(check["late_delivery_risk"].mean() * 100, 1), "%")

Saved shape: (172765, 22)
Reloaded shape: (172765, 22)
Same columns: True
Date type: datetime64[us]
First order: 2015-01-01 00:00:00
Last order: 2018-01-31 23:38:00
Orders: 62897
Late share: 57.3 %
